# India DPDP Act - GraphRAG Assistant with Human-in-the-Loop Review

A production-shaped demo of a policy knowledge-graph + retrieval-augmented assistant, scoped to a
single law (India's Digital Personal Data Protection Act, 2023) so the full pipeline is easy to run,
explain, and verify end to end. Runs entirely in Google Colab, fully open source, no API key needed
anywhere, no paid services.

## What this demonstrates

A multi-country policy assistant needs: a **knowledge graph** connecting laws to their obligations,
rights, and penalties; a **vector store** for semantic search over the legal text; an **AI agent**
that retrieves and generates grounded answers; a **human-in-the-loop review gate** so nothing
unverified reaches an end user; and a real, production-quality **interface** for people to actually
use it - not a default widget library, a custom-built chat application with streaming responses,
a live review dashboard, and a citation viewer.

## Architecture

```
Official DPDP Act PDF (fetched live from meity.gov.in)
        |
        v
Parse into 44 numbered sections (regex, validated against the real text)
        |
        v
  Rule-based tagging: category (Obligation / Right / Penalty / Definition) + confidence
        |
  +-----+-----------------------------------+
  |                                         |
  v                                         v
Auto-approved                      Held for HUMAN REVIEW
(safe category, high confidence)   (touches Obligation/Penalty, or low confidence)
  |                                         |
  v                                         v
Written to Qdrant (vectors)         Sits in a review queue until a human
+ Neo4j (graph)                     approves/rejects it in the live dashboard -
  |                                  only then is it written to Qdrant + Neo4j too
  |
  v
FastAPI backend (/ask streams NDJSON, /pending-review, /approve-review-item, /stats, /section/{id})
  |
  v
On a question: vector search (Qdrant) + graph context (Neo4j) retrieved,
then Ollama (qwen2.5:3b, local, no API key) generates a cited answer, TOKEN-STREAMED
back to the browser - UNLESS the question itself is high-risk (penalty/obligation/
cross-border), in which case it's held for human review instead of answered directly
  |
  v
Custom HTML/CSS/JS chat application (served by the SAME FastAPI process)
  --->  Cloudflare quick tunnel  --->  one public HTTPS URL
```

## Tech stack (all open source, all free, no signup required anywhere)

| Component | Tool | Role |
|---|---|---|
| Knowledge graph | Neo4j (Community) | Sections linked to obligations, rights, penalties, definitions |
| Vector store | Qdrant (in-memory) | Semantic search over section text |
| Embeddings | `sentence-transformers` (`all-MiniLM-L6-v2`) | Turns section text and questions into vectors |
| Local LLM | Ollama, running `qwen2.5:3b` | Generates the final answer text, streamed token-by-token |
| Backend | FastAPI + Uvicorn | REST + streaming API, also serves the frontend (one process, one port) |
| Frontend | Hand-built HTML/CSS/JS (no framework, no build step) | Production-style chat UI: streaming replies, live review queue, citation modal |
| Public URL | `cloudflared` quick tunnel | Free, no signup, no interstitial warning page |
| Source data | Official Gazette PDF, Ministry of Electronics & IT (MeitY) | Fetched live at runtime, not hardcoded |

**Why the frontend was rebuilt instead of using a widget library:** the earlier version of this demo
used Streamlit, which is great for quick internal tools but looks and behaves like a data-app widget
library, not a product. This version serves a purpose-built single-page app - streamed responses
(text appears as the model generates it, like a real chat product), a live-updating review sidebar,
and a citation system where clicking a cited section opens the actual Act text - all still 100% open
source (vanilla JS, no paid fonts/assets/APIs) and still just files written from this notebook.

## Human-in-the-loop: two checkpoints, and why they matter (with a real example)

**Checkpoint 1 - before anything enters the graph/vector store.** Every parsed section is tagged by
chapter and title (e.g. Chapter II = "Obligations of Data Fiduciary"). Any section touching an
Obligation or Penalty, or with a low parse-confidence score, is held out of Qdrant/Neo4j entirely
until a human approves it through the review dashboard. This is "the model proposes, a human
disposes" applied to policy data: a wrong reading of a penalty clause is worse than a missing one.

**Checkpoint 2 - before a generated answer is shown.** Questions containing high-risk keywords
(penalty, obligation, breach, cross-border transfer) never get an auto-generated answer - they're
returned as `pending_review` with the retrieved context and citations attached, so a human decides
before anything resembling advice reaches an end user.

**Why this isn't just theoretical - an example from testing this exact pipeline:** asking the
one-word query *"rules"* correctly retrieved Section 41 (citations are computed separately from the
generated text, via vector search), but the local model's generated sentence mislabeled it as
"Section 42" while paraphrasing. Asking the more specific *"what does section 41 require"* produced
an accurate answer citing the same section correctly. This is a real, observed limitation of small
local LLMs: they can retrieve the right source and still misstate a detail while summarizing it -
which is exactly why citations are surfaced independently of the generated prose (and shown as
clickable chips that open the real section text), and why high-risk answers are gated behind human
review rather than trusted automatically.

## Groundedness guardrail (why the assistant refuses out-of-scope questions)

Early testing surfaced a real failure mode worth documenting rather than hiding: asking an
out-of-scope question like *"What is GDPR?"* returned a fluent, confident-sounding answer with
**no citations at all** - because Qdrant always returns its nearest top-k vectors even when nothing
is actually relevant, and the local model filled the gap with its own general training knowledge
instead of admitting the topic wasn't covered. That's the opposite of "grounded, citation-backed
answers," so it's fixed at the retrieval layer, not papered over with prompt wording alone:

- Qdrant now applies a **hard similarity cutoff** (`HARD_CUTOFF` in `dpdp_config.py`) server-side,
  so genuinely unrelated questions retrieve *zero* context instead of "the least-bad top 3."
- With zero context, the LLM is never called at all - the API returns a clear
  `"I couldn't find anything about this in the DPDP Act, 2023"` message instead of generating text.
- Matches that clear the hard cutoff but fall below a higher `SIMILARITY_THRESHOLD` are still
  answered, but flagged as low-confidence in the UI, rather than either silently guessing or
  silently refusing.
- The system prompt sent to the model explicitly forbids using outside knowledge, even on topics
  it clearly recognizes (GDPR, CCPA, etc.), and gives it an exact refusal sentence to fall back on.

`HARD_CUTOFF` and `SIMILARITY_THRESHOLD` are heuristics tuned on `all-MiniLM-L6-v2` cosine
similarity for this specific corpus - if you see either too many false "not found" answers on
clearly in-scope questions, or any hallucinated one on an out-of-scope question, that's the first
place to adjust (the retrieval log line in `dpdp.main` prints every query's top score, which is the
fastest way to pick a better threshold from real usage).

## How to run

Run All, top to bottom. Cells under "Install" and "Start Neo4j / Ollama" take a few minutes,
one-time per session. The backend writes five source files (`dpdp_config.py`, `dpdp_ingest.py`,
`dpdp_stores.py`, `main.py`, and the `static/` frontend) directly to disk via `%%writefile` - no
string-escaping risk, and you can open/edit any of them afterward like normal files. The wait/test
cell should complete in well under a minute, since no LLM calls happen until a question is actually
asked. The last cell prints a public HTTPS URL (`*.trycloudflare.com`) - open it directly, no
password or signup needed.

## Performance notes

Generation runs on Ollama's CPU inference by default on Colab's free tier, which is the main source
of per-question latency. Three things help:
- **Switch to a GPU runtime** (Runtime -> Change runtime type -> T4 GPU) before running - Ollama
  auto-detects CUDA, typically a 5-10x speedup, zero code changes needed.
- The model is **pre-warmed** right after it's pulled, so the first real question isn't also paying
  the one-time cost of loading weights into memory.
- Generation is capped (`num_predict`), the model is kept loaded between calls (`keep_alive`),
  identical repeat questions are served from an in-memory cache, and answers are **streamed** token
  by token so the perceived latency (time to first visible text) is much lower even when total
  generation time is unchanged.

## Known limitations (worth stating plainly, not hiding)

- **Tagging is rule-based (chapter/title keyword matching), not LLM-based**, for reliability and
  speed in a live demo - see "Where this would grow" at the end for how a real extraction agent
  would replace this.
- **Only the original 2023 Act text is covered** - the Digital Personal Data Protection Rules, 2025
  (notified 13 November 2025) are not included.
- **Neo4j and Qdrant are ephemeral** - both live inside this Colab session and are wiped when it ends.
- **The rate limiter and answer cache are in-memory, single-process** - fine for a demo behind one
  tunnel URL, not a substitute for Redis/a real gateway at real scale.
- **Small local LLM can misstate details while summarizing**, even when citing the correct source -
  demonstrated above. Citations are shown independently and are clickable so this is always
  checkable in the UI itself.

## Source

Official Gazette of India, Ministry of Law and Justice, 11 August 2023, published by the Ministry of
Electronics and Information Technology (MeitY):
https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf


## 1. Install everything (system + Python + tunnel tooling)

In [ ]:
print("Installing system dependencies (zstd, OpenJDK for Neo4j, Ollama)...")
!apt-get update -qq
!apt-get install -y -qq zstd openjdk-17-jre-headless

!curl -fsSL https://ollama.com/install.sh | sh

print()
print("Installing cloudflared (for the public tunnel - free, no signup, no interstitial page)...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print()
print("Installing Python libraries...")
!pip install -q pypdf sentence-transformers qdrant-client neo4j ollama requests pydantic fastapi uvicorn


## 2. Start Neo4j

Downloads directly from `dist.neo4j.org` (not the `neo4j.com/artifact.php` redirect) and waits until
the database actually accepts connections before moving on.

In [ ]:
import subprocess
import time
import os

NEO4J_VERSION = "5.15.0"
NEO4J_DIR = f"/content/neo4j-community-{NEO4J_VERSION}"

print("Downloading Neo4j...")
subprocess.run(
    ["wget", "-q", f"https://dist.neo4j.org/neo4j-community-{NEO4J_VERSION}-unix.tar.gz", "-O", "neo4j.tar.gz"],
    check=True,
)
subprocess.run(["tar", "-xzf", "neo4j.tar.gz"], check=True)
os.environ["NEO4J_HOME"] = NEO4J_DIR

print("Setting initial password...")
subprocess.run([f"{NEO4J_DIR}/bin/neo4j-admin", "dbms", "set-initial-password", "password"], check=True)

print("Starting Neo4j (background)...")
subprocess.Popen([f"{NEO4J_DIR}/bin/neo4j", "start"])


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "password")


def wait_for_neo4j(max_attempts=30, delay_seconds=5):
    for attempt in range(max_attempts):
        try:
            driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
            driver.verify_connectivity()
            print(f"Neo4j is up (took about {attempt * delay_seconds}s).")
            driver.close()
            return
        except Exception:
            time.sleep(delay_seconds)
    raise RuntimeError("Neo4j did not become ready in time - re-run this cell, it may just need longer.")


wait_for_neo4j()


## 3. Start Ollama and pull the local model

Only used for answering questions later (one call per question) - not for startup ingestion, so a
slow pull here doesn't block the backend from becoming healthy.

**On speed:** if you're on Colab's free CPU-only runtime, `qwen2.5:3b` generation can take a while
per question. The single biggest speedup available for free is switching to a GPU runtime -
**Runtime -> Change runtime type -> T4 GPU** - before running this notebook. Ollama detects CUDA
automatically and uses it with no code changes needed. The cell below also pre-warms the model
(loads it into memory right now) so your *first* real question during the demo isn't slow too.

In [ ]:
print("Starting Ollama server (background)...")
subprocess.Popen(["ollama", "serve"])
time.sleep(10)

print("Pulling qwen2.5:3b (open source, ~2GB, one-time download)...")
subprocess.run(["ollama", "pull", "qwen2.5:3b"], check=True)

print("Pre-warming the model (loading it into memory now, so the first real question is fast)...")
import ollama as _ollama_warmup
_ollama_warmup.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Say ready."}],
    options={"num_predict": 5},
    keep_alive="30m",
)
print("Ollama ready and warmed up.")


## 4. Quick sanity check - live fetch + parse (no LLM, no wait)

Confirms the PDF fetch and section splitter work. Same sequential-section-number splitter used in
`dpdp_ingest.py` below - validated to correctly find all 44 sections in the real Act text.

In [ ]:
import re
import requests
from pypdf import PdfReader

DPDP_PDF_URL = "https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf"

response = requests.get(DPDP_PDF_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
response.raise_for_status()
with open("dpdp_act_2023.pdf", "wb") as f:
    f.write(response.content)

reader = PdfReader("dpdp_act_2023.pdf")
raw_text = ""
for page in reader.pages:
    raw_text += (page.extract_text() or "") + "\n"


def find_section_start(text, number, search_from):
    pattern = re.compile(rf"(?:^|\n){number}\.\s")
    match = pattern.search(text, search_from)
    return match.start() if match else -1


positions = {}
cursor = 0
for n in range(1, 45):
    pos = find_section_start(raw_text, n, cursor)
    if pos != -1:
        positions[n] = pos
        cursor = pos + 1

missing = [n for n in range(1, 45) if n not in positions]
print(f"Fetched {len(response.content)} bytes. Parsed {len(positions)} of 44 sections.")
print("Missing:", missing if missing else "none - all 44 found")


## 5. Write the backend config module (`dpdp_config.py`)

Every constant - chapter map, section titles, thresholds, model names - lives in one place, shared
by the ingestion pipeline and the API layer, instead of being duplicated or scattered through a
single giant script.

In [ ]:
%%writefile dpdp_config.py
"""
Central configuration for the DPDP Act GraphRAG service.

Keeping every constant in one importable module (instead of scattered across
main.py) means the ingestion pipeline, the API layer, and any future batch
job can share the exact same source of truth.
"""

import os

# ---------------------------------------------------------------------------
# Source data
# ---------------------------------------------------------------------------
DPDP_PDF_URL = (
    "https://www.meity.gov.in/static/uploads/2024/06/"
    "2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf"
)
LOCAL_PDF_PATH = "dpdp_act_2023.pdf"
MAX_SECTIONS = 44

# ---------------------------------------------------------------------------
# Infra endpoints (all local, all open source, no keys)
# ---------------------------------------------------------------------------
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = ("neo4j", os.environ.get("NEO4J_PASSWORD", "password"))
QDRANT_COLLECTION = "dpdp_act_chunks"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:3b")
OLLAMA_NUM_PREDICT = 260
OLLAMA_NUM_CTX = 2048  # bumped to fit RETRIEVAL_TOP_K=5 sections of context comfortably
OLLAMA_KEEP_ALIVE = "30m"

# ---------------------------------------------------------------------------
# Human-in-the-loop thresholds
# ---------------------------------------------------------------------------
CONFIDENCE_THRESHOLD = 0.85
SENSITIVE_CATEGORIES = ("obligations", "penalties")
HIGH_RISK_QUERY_KEYWORDS = (
    "penalty", "fine", "punish", "breach", "obligation", "must",
    "cross-border", "cross border", "transfer outside",
)

# ---------------------------------------------------------------------------
# API behaviour
# ---------------------------------------------------------------------------
RETRIEVAL_TOP_K = 5

# --- Groundedness / anti-hallucination controls ---
# Cosine similarity (all-MiniLM-L6-v2) between the query and a genuinely
# relevant DPDP Act section is typically 0.30+. Below HARD_CUTOFF we treat
# the result as noise (Qdrant always returns *something* if the collection
# is non-empty, even for totally unrelated queries like "What is GDPR?").
# Between HARD_CUTOFF and SIMILARITY_THRESHOLD we still show the answer but
# flag it as low-confidence, since a false "not found" is far safer than a
# hallucinated legal answer, but we don't want to over-refuse borderline
# in-scope questions either. Tune these by checking the logged scores
# (dpdp.main logs every retrieval's top score) against real demo questions.
SIMILARITY_THRESHOLD = 0.30
HARD_CUTOFF = 0.15
CONTEXT_CHAR_LIMIT = 800  # per-section chars included in the LLM prompt

RATE_LIMIT_REQUESTS_PER_MINUTE = 20  # simple in-memory per-IP throttle, no Redis needed
PDF_FETCH_RETRIES = 3
PDF_FETCH_BACKOFF_SECONDS = 3

CHAPTER_MAP = [
    ("I", "PRELIMINARY", 1, 3),
    ("II", "OBLIGATIONS OF DATA FIDUCIARY", 4, 10),
    ("III", "RIGHTS AND DUTIES OF DATA PRINCIPAL", 11, 15),
    ("IV", "SPECIAL PROVISIONS", 16, 17),
    ("V", "DATA PROTECTION BOARD OF INDIA", 18, 26),
    ("VI", "POWERS, FUNCTIONS AND PROCEDURE TO BE FOLLOWED BY BOARD", 27, 28),
    ("VII", "APPEAL AND ALTERNATE DISPUTE RESOLUTION", 29, 32),
    ("VIII", "PENALTIES AND ADJUDICATION", 33, 34),
    ("IX", "MISCELLANEOUS", 35, 44),
]

SECTION_TITLES = {
    1: "Short title and commencement", 2: "Definitions", 3: "Application of Act",
    4: "Grounds for processing personal data", 5: "Notice", 6: "Consent",
    7: "Certain legitimate uses", 8: "General obligations of Data Fiduciary",
    9: "Processing of personal data of children",
    10: "Additional obligations of Significant Data Fiduciary",
    11: "Right to access information about personal data",
    12: "Right to correction and erasure of personal data",
    13: "Right of grievance redressal", 14: "Right to nominate",
    15: "Duties of Data Principal", 16: "Processing of personal data outside India",
    17: "Exemptions", 18: "Establishment of Board",
    19: "Composition and qualifications for appointment of Chairperson and Members",
    20: "Salary, allowances payable to and term of office",
    21: "Disqualifications for appointment and continuation as Chairperson and Members",
    22: "Resignation by Members and filling of vacancy", 23: "Proceedings of Board",
    24: "Officers and employees of Board", 25: "Members and officers to be public servants",
    26: "Powers of Chairperson", 27: "Powers and functions of Board",
    28: "Procedure to be followed by Board", 29: "Appeal to Appellate Tribunal",
    30: "Orders passed by Appellate Tribunal to be executable as decree",
    31: "Alternate dispute resolution", 32: "Voluntary undertaking", 33: "Penalties",
    34: "Crediting sums realised by way of penalties to Consolidated Fund of India",
    35: "Protection of action taken in good faith", 36: "Power to call for information",
    37: "Power of Central Government to issue directions", 38: "Consistency with other laws",
    39: "Bar of jurisdiction", 40: "Power to make rules",
    41: "Laying of rules and certain notifications", 42: "Power to amend Schedule",
    43: "Power to remove difficulties", 44: "Amendments to certain Acts",
}


def chapter_for_section(section_number: int) -> str:
    for roman, title, start, end in CHAPTER_MAP:
        if start <= section_number <= end:
            return f"Chapter {roman} - {title}"
    return "Unknown"


## 6. Write the ingestion pipeline (`dpdp_ingest.py`)

Pure functions: fetch PDF -> split into sections -> rule-based tag -> confidence score. No side
effects on Qdrant/Neo4j here, which keeps this piece independently testable and is exactly the
module a future LLM-based extraction agent would replace (see the closing section of this
notebook).

In [ ]:
%%writefile dpdp_ingest.py
"""
Ingestion pipeline: fetch the official PDF -> split into numbered sections ->
tag each section with rule-based categories -> score a confidence heuristic.

Pure functions with no side effects on the stores, so they're easy to unit
test in isolation (see the sanity-check cell in the notebook) and easy to
swap out later for a real LLM extraction agent (see README "grows into").
"""

import logging
import os
import re
import time

import requests

from dpdp_config import (
    CHAPTER_MAP, DPDP_PDF_URL, LOCAL_PDF_PATH, SECTION_TITLES,
    SENSITIVE_CATEGORIES, chapter_for_section,
)

logger = logging.getLogger("dpdp.ingest")

_SECTION_START_RE_CACHE: dict[int, "re.Pattern[str]"] = {}


def _section_start_pattern(number: int) -> "re.Pattern[str]":
    if number not in _SECTION_START_RE_CACHE:
        _SECTION_START_RE_CACHE[number] = re.compile(rf"(?:^|\n){number}\.\s")
    return _SECTION_START_RE_CACHE[number]


def fetch_act_pdf(
    dest_path: str = LOCAL_PDF_PATH,
    url: str = DPDP_PDF_URL,
    max_retries: int = 3,
    backoff_seconds: float = 3.0,
) -> str:
    """Download the official Gazette PDF once and cache it on disk.

    Retries with backoff (government sites occasionally 5xx or time out),
    and validates the response actually starts with a PDF magic number -
    a common silent-failure mode is the server returning a 200 OK HTML
    error/maintenance page instead of the real file, which would otherwise
    crash confusingly deep inside pypdf instead of here with a clear cause."""
    if os.path.exists(dest_path):
        with open(dest_path, "rb") as f:
            if f.read(4) == b"%PDF":
                logger.info("Using cached, validated PDF at %s", dest_path)
                return dest_path
        logger.warning("Cached file at %s is not a valid PDF - re-fetching", dest_path)
        os.remove(dest_path)

    last_error: Exception | None = None
    for attempt in range(1, max_retries + 1):
        try:
            logger.info("Fetching DPDP Act PDF from %s (attempt %d/%d)", url, attempt, max_retries)
            response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
            response.raise_for_status()
            if not response.content.startswith(b"%PDF"):
                raise ValueError(
                    "Response did not look like a PDF (got a webpage instead - the source "
                    "URL may be temporarily down or redirecting to an error page)."
                )
            with open(dest_path, "wb") as f:
                f.write(response.content)
            logger.info("Saved %d bytes to %s", len(response.content), dest_path)
            return dest_path
        except Exception as exc:
            last_error = exc
            logger.warning("PDF fetch attempt %d failed: %s", attempt, exc)
            if attempt < max_retries:
                time.sleep(backoff_seconds * attempt)

    raise RuntimeError(
        f"Could not fetch the DPDP Act PDF from {url} after {max_retries} attempts: {last_error}"
    )


def extract_text(pdf_path: str) -> str:
    from pypdf import PdfReader

    reader = PdfReader(pdf_path)
    return "\n".join((page.extract_text() or "") for page in reader.pages)


def split_into_sections(text: str, max_section: int = 44) -> list[dict]:
    """Split the raw Act text into numbered sections using sequential-anchor
    matching (each section number is searched for only after the previous
    one), which avoids false positives from numbers inside body text."""
    positions: dict[int, int] = {}
    cursor = 0
    for n in range(1, max_section + 1):
        match = _section_start_pattern(n).search(text, cursor)
        if not match:
            continue
        positions[n] = match.start()
        cursor = match.start() + 1

    found_numbers = sorted(positions.keys())
    parsed = []
    for i, n in enumerate(found_numbers):
        start = positions[n]
        end = positions[found_numbers[i + 1]] if i + 1 < len(found_numbers) else start + 4000
        body = text[start:end].strip()
        parsed.append({
            "id": f"S{n}",
            "number": n,
            "chapter": chapter_for_section(n),
            "title": SECTION_TITLES.get(n, f"Section {n}"),
            "raw_text": body,
            "source_url": DPDP_PDF_URL,
        })
    return parsed


_DEFINITION_RE = re.compile(r'["\u201c]([^"\u201d]{2,60})["\u201d]\s+means')


def tag_entities(section: dict) -> dict:
    title_lower = section["title"].lower()
    chapter_upper = section["chapter"].upper()
    entities = {"obligations": [], "rights": [], "penalties": [], "definitions": []}

    if "right" in title_lower:
        entities["rights"].append(section["title"])
    if "obligation" in title_lower or "duties" in title_lower or "OBLIGATIONS" in chapter_upper:
        entities["obligations"].append(section["title"])
    if "penalt" in title_lower or "PENALTIES" in chapter_upper:
        entities["penalties"].append(section["title"])
    if section["number"] == 2:
        entities["definitions"] = _DEFINITION_RE.findall(section["raw_text"])[:10]

    return entities


def estimate_confidence(section: dict) -> float:
    length = len(section["raw_text"])
    if length < 40:
        return 0.4
    score = 0.75 + min(length, 1200) / 1200 * 0.2
    return round(min(score, 0.97), 2)


def is_sensitive(entities: dict) -> bool:
    return any(len(entities.get(cat, [])) > 0 for cat in SENSITIVE_CATEGORIES)


def build_tagged_sections(pdf_path: str, max_section: int = 44) -> list[dict]:
    """End-to-end: parse -> tag -> score. Returns sections ready for the
    review gate; does not touch Qdrant/Neo4j (see dpdp_stores.py)."""
    raw_text = extract_text(pdf_path)
    sections = split_into_sections(raw_text, max_section)
    for section in sections:
        section["entities"] = tag_entities(section)
        section["confidence"] = estimate_confidence(section)
        section["sensitive"] = is_sensitive(section["entities"])
    if len(sections) < max_section:
        missing = sorted(set(range(1, max_section + 1)) - {s["number"] for s in sections})
        logger.warning(
            "Only parsed %d/%d sections - missing section numbers: %s. "
            "The source PDF's layout may have changed; check split_into_sections().",
            len(sections), max_section, missing,
        )
    else:
        logger.info("Parsed and tagged %d/%d sections", len(sections), max_section)
    return sections


## 7. Write the vector/graph stores + review queue (`dpdp_stores.py`)

`VectorGraphStore` owns the embedder, the Qdrant client, and the Neo4j driver behind one interface.
`ReviewQueue` is a small thread-safe in-memory queue (Colab has no Redis, and doesn't need one for a
single-process demo). `AnswerCache` and `RateLimiter` are the same idea applied to repeat questions
and per-client throttling.

In [ ]:
%%writefile dpdp_stores.py
"""
Thin, typed wrappers around Qdrant and Neo4j, plus the in-memory
human-review queue and answer cache.

Isolating these behind classes (instead of module-level globals scattered
through main.py) makes startup/shutdown explicit and makes the whole thing
testable without a running FastAPI app.
"""

import logging
import threading
import uuid
from datetime import datetime, timezone
from typing import Optional

from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer

from dpdp_config import EMBEDDING_DIM, EMBEDDING_MODEL, NEO4J_AUTH, NEO4J_URI, QDRANT_COLLECTION

logger = logging.getLogger("dpdp.stores")


class VectorGraphStore:
    """Owns the embedder, the Qdrant client, and the Neo4j driver."""

    def __init__(self) -> None:
        self.embedder = SentenceTransformer(EMBEDDING_MODEL)
        self.qdrant = QdrantClient(":memory:")
        self.neo4j = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
        self._collection_ready = False

    def ping_neo4j(self) -> bool:
        try:
            self.neo4j.verify_connectivity()
            return True
        except Exception:
            return False

    def ensure_collection(self) -> None:
        if self._collection_ready:
            return
        if not self.qdrant.collection_exists(QDRANT_COLLECTION):
            self.qdrant.create_collection(
                collection_name=QDRANT_COLLECTION,
                vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
            )
        self._collection_ready = True

    def embed(self, text: str) -> list[float]:
        return self.embedder.encode(text).tolist()

    def commit_section(self, section: dict) -> None:
        """Write one approved section into both Qdrant (vector) and Neo4j
        (graph). Called either automatically (auto-approved sections) or
        from the /approve-review-item endpoint (human-approved sections)."""
        self.ensure_collection()
        vector = self.embed(section["raw_text"])
        self.qdrant.upsert(
            collection_name=QDRANT_COLLECTION,
            points=[PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={
                    "kg_node_id": section["id"],
                    "title": section["title"],
                    "chapter": section["chapter"],
                    "text": section["raw_text"],
                    "source_url": section["source_url"],
                },
            )],
        )

        with self.neo4j.session() as session:
            session.run(
                "MERGE (s:Section {id: $id}) SET s.title = $title, s.source_url = $url",
                id=section["id"], title=section["title"], url=section["source_url"],
            )
            entities = section["entities"]
            for label, items in (
                ("Obligation", entities["obligations"]), ("Right", entities["rights"]),
                ("Penalty", entities["penalties"]), ("Definition", entities["definitions"]),
            ):
                for name in items:
                    node_id = f"{label.lower()}::{name}"[:200]
                    session.run(
                        f"MERGE (e:`{label}` {{id: $id}}) SET e.name = $name",
                        id=node_id, name=name,
                    )
                    session.run(
                        "MATCH (s:Section {id: $sid}), (e {id: $eid}) MERGE (s)-[:MENTIONS]->(e)",
                        sid=section["id"], eid=node_id,
                    )
        logger.info("Committed %s to Qdrant + Neo4j", section["id"])

    def search(self, query_vector: list[float], top_k: int, score_threshold: float | None = None) -> list:
        """Vector search. When score_threshold is given, Qdrant filters out
        weakly-related results server-side instead of always returning the
        nearest top_k regardless of how irrelevant they actually are - this
        is what stops the assistant from confidently answering questions
        that have nothing to do with the indexed DPDP Act text."""
        self.ensure_collection()
        return self.qdrant.query_points(
            collection_name=QDRANT_COLLECTION,
            query=query_vector,
            limit=top_k,
            score_threshold=score_threshold,
        ).points

    def graph_context(self, section_id: str) -> list[dict]:
        cypher = """
        MATCH (s:Section {id: $sid})-[:MENTIONS]->(related)
        RETURN labels(related)[0] AS type, related.name AS name
        """
        with self.neo4j.session() as session:
            return session.run(cypher, sid=section_id).data()

    def indexed_count(self) -> int:
        self.ensure_collection()
        return self.qdrant.count(collection_name=QDRANT_COLLECTION).count


class ReviewQueue:
    """Thread-safe in-memory human-in-the-loop queue. One entry per section."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._entries: dict[str, dict] = {}
        self._sections: dict[str, dict] = {}  # full section payload, for approve-time commit

    def register(self, section: dict, needs_review: bool) -> None:
        with self._lock:
            self._sections[section["id"]] = section
            self._entries[section["id"]] = {
                "section_id": section["id"],
                "title": section["title"],
                "chapter": section["chapter"],
                "confidence": section["confidence"],
                "sensitive": section["sensitive"],
                "status": "pending_review" if needs_review else "auto_approved",
                "reviewed_by": None,
                "reviewed_at": None,
            }

    def pending(self) -> list[dict]:
        with self._lock:
            return [e for e in self._entries.values() if e["status"] == "pending_review"]

    def all_entries(self) -> list[dict]:
        with self._lock:
            return list(self._entries.values())

    def get_section(self, section_id: str) -> Optional[dict]:
        with self._lock:
            return self._sections.get(section_id)

    def decide(self, section_id: str, decision: str, reviewer: str) -> dict:
        with self._lock:
            if section_id not in self._entries:
                raise KeyError(section_id)
            entry = self._entries[section_id]
            entry["status"] = "approved" if decision == "approve" else "rejected"
            entry["reviewed_by"] = reviewer
            entry["reviewed_at"] = datetime.now(timezone.utc).isoformat()
            return dict(entry)

    def stats(self) -> dict:
        with self._lock:
            values = list(self._entries.values())
        counts = {"total": len(values)}
        for status in ("auto_approved", "pending_review", "approved", "rejected"):
            counts[status] = sum(1 for e in values if e["status"] == status)
        return counts


class AnswerCache:
    """Simple exact-match, size-capped cache so repeat questions skip
    retrieval + generation entirely."""

    def __init__(self, max_entries: int = 200) -> None:
        self._lock = threading.Lock()
        self._data: dict[str, dict] = {}
        self._max_entries = max_entries

    @staticmethod
    def _key(query: str) -> str:
        return query.strip().lower()

    def get(self, query: str) -> Optional[dict]:
        with self._lock:
            return self._data.get(self._key(query))

    def set(self, query: str, value: dict) -> None:
        with self._lock:
            if len(self._data) >= self._max_entries:
                self._data.pop(next(iter(self._data)))
            self._data[self._key(query)] = value


class RateLimiter:
    """Fixed-window per-client throttle, entirely in-memory — no Redis
    needed for a single-process Colab demo."""

    def __init__(self, limit_per_minute: int) -> None:
        self._lock = threading.Lock()
        self._limit = limit_per_minute
        self._windows: dict[str, tuple[int, float]] = {}

    def allow(self, client_key: str) -> bool:
        import time
        now = time.time()
        with self._lock:
            count, window_start = self._windows.get(client_key, (0, now))
            if now - window_start >= 60:
                count, window_start = 0, now
            count += 1
            self._windows[client_key] = (count, window_start)
            return count <= self._limit


## 8. Write the FastAPI backend (`main.py`)

Ties the modules above together. `/ask` now **streams** the answer back as newline-delimited JSON
(one token per line, then a final summary line with citations) instead of blocking until the whole
answer is generated - this is what lets the UI show text appearing live, like a real chat product.
The same process also serves the frontend (`static/`) at `/`, so the whole app is one Uvicorn
process and one port.

In [ ]:
%%writefile main.py
"""
DPDP Act GraphRAG service — API + frontend in one process.

Startup: fetch the official PDF, parse into sections, tag with rule-based
categories (fast, no LLM), gate through human review, write approved
sections into Qdrant + Neo4j. Ollama is only called inside /ask, once per
question, and streamed back token-by-token so the UI can render it like a
real chat product.

Everything here is open source and runs with no API key: FastAPI, Qdrant
(in-memory), Neo4j (local), sentence-transformers, Ollama (local model).
"""

import json
import logging
import time
from contextlib import asynccontextmanager
from typing import Optional

import ollama
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field

import dpdp_config as cfg
from dpdp_ingest import build_tagged_sections, fetch_act_pdf
from dpdp_stores import AnswerCache, RateLimiter, ReviewQueue, VectorGraphStore

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("dpdp.main")

store = VectorGraphStore()
review_queue = ReviewQueue()
answer_cache = AnswerCache()
rate_limiter = RateLimiter(cfg.RATE_LIMIT_REQUESTS_PER_MINUTE)

_ingestion_done = False
_ingestion_error: Optional[str] = None


def run_ingestion() -> None:
    """Idempotent: safe to call from startup and defensively from /ask.
    On failure, records a clear error (surfaced via /health) instead of
    leaving the frontend stuck on "Ingesting Act text..." forever - and
    a later call (e.g. the next /ask) will retry rather than staying
    permanently broken for the rest of the Colab session."""
    global _ingestion_done, _ingestion_error
    if _ingestion_done:
        return

    try:
        pdf_path = fetch_act_pdf(
            max_retries=cfg.PDF_FETCH_RETRIES, backoff_seconds=cfg.PDF_FETCH_BACKOFF_SECONDS,
        )
        sections = build_tagged_sections(pdf_path, cfg.MAX_SECTIONS)

        auto_approved = 0
        for section in sections:
            needs_review = section["sensitive"] or section["confidence"] < cfg.CONFIDENCE_THRESHOLD
            review_queue.register(section, needs_review)
            if not needs_review:
                store.commit_section(section)
                auto_approved += 1

        logger.info(
            "Ingestion complete: %d/%d auto-approved and indexed, %d pending human review.",
            auto_approved, len(sections), len(sections) - auto_approved,
        )
        _ingestion_error = None
        _ingestion_done = True
    except Exception as exc:
        logger.exception("Ingestion failed")
        _ingestion_error = str(exc)
        # _ingestion_done stays False so the next call (health poll or /ask) retries


@asynccontextmanager
async def lifespan(_: FastAPI):
    run_ingestion()
    yield


app = FastAPI(title="DPDP Act GraphRAG API", version="2.0.0", lifespan=lifespan)
app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)


# ---------------------------------------------------------------------------
# Schemas
# ---------------------------------------------------------------------------
class QueryRequest(BaseModel):
    query: str = Field(..., min_length=1, max_length=500)

    def clean_query(self) -> str:
        cleaned = self.query.strip()
        if not cleaned:
            raise HTTPException(status_code=400, detail="Query cannot be empty or whitespace-only.")
        return cleaned


class ReviewDecision(BaseModel):
    section_id: str
    decision: str
    reviewer: str = "demo_reviewer"


# ---------------------------------------------------------------------------
# Health / observability
# ---------------------------------------------------------------------------
@app.get("/health")
def health():
    return {
        "status": "ok",
        "ingested": _ingestion_done,
        "ingestion_error": _ingestion_error,
        "neo4j_connected": store.ping_neo4j(),
        "sections_indexed": store.indexed_count() if _ingestion_done else 0,
    }


@app.get("/stats")
def stats():
    counts = review_queue.stats()
    return {
        "sections_total": counts["total"],
        "auto_approved": counts["auto_approved"],
        "pending_review": counts["pending_review"],
        "human_approved": counts["approved"],
        "rejected": counts["rejected"],
        "indexed_in_qdrant": store.indexed_count() if _ingestion_done else 0,
    }


# ---------------------------------------------------------------------------
# Human-in-the-loop review
# ---------------------------------------------------------------------------
@app.get("/pending-review")
def pending_review():
    return review_queue.pending()


@app.get("/section/{section_id}")
def get_section(section_id: str):
    section = review_queue.get_section(section_id)
    if not section:
        raise HTTPException(status_code=404, detail=f"No such section {section_id}")
    return {
        "id": section["id"], "title": section["title"], "chapter": section["chapter"],
        "text": section["raw_text"], "source_url": section["source_url"],
        "entities": section["entities"],
    }


@app.post("/approve-review-item")
def approve_review_item(decision: ReviewDecision):
    if decision.decision not in ("approve", "reject"):
        raise HTTPException(status_code=400, detail="decision must be 'approve' or 'reject'")
    try:
        entry = review_queue.decide(decision.section_id, decision.decision, decision.reviewer)
    except KeyError:
        raise HTTPException(status_code=404, detail=f"No such section {decision.section_id}")

    if decision.decision == "approve":
        store.commit_section(review_queue.get_section(decision.section_id))

    return {"section_id": decision.section_id, "new_status": entry["status"]}


# ---------------------------------------------------------------------------
# Chat
# ---------------------------------------------------------------------------
def query_is_high_risk(query: str) -> bool:
    lowered = query.lower()
    return any(keyword in lowered for keyword in cfg.HIGH_RISK_QUERY_KEYWORDS)


def _ndjson(obj: dict) -> str:
    return json.dumps(obj) + "\n"


SYSTEM_PROMPT = (
    "You are a legal-reference assistant for India's Digital Personal Data Protection "
    "Act, 2023 (DPDP Act) only. Follow these rules strictly:\n"
    "1. Answer using ONLY the numbered [S<n>] context sections provided below. Never use "
    "any outside knowledge about data protection law, even if you recognize the topic "
    "(e.g. GDPR, CCPA, or general legal concepts NOT present in the given context).\n"
    "2. If the provided context does not contain enough information to answer the "
    "question, reply with exactly: \"I don't have information about this in the DPDP "
    "Act, 2023 based on what's currently indexed.\" Do not guess or generalize.\n"
    "3. Keep answers concise (2-4 sentences), plain language, and reference section "
    "numbers only from the context given - never invent a section number.\n"
    "4. This is informational only, not legal advice."
)


def _retrieve(query: str) -> tuple[str, list[str], list[dict], float]:
    """Vector search + graph enrichment.

    Returns (context_str, citation_ids, citation_meta, top_score). Results
    below cfg.HARD_CUTOFF are filtered out by Qdrant server-side (see
    VectorGraphStore.search) - this is what stops an unrelated question
    like "What is GDPR?" from being handed *any* context at all, which is
    the root fix for the model answering from outside knowledge instead of
    admitting the topic isn't covered."""
    query_vector = store.embed(query)
    results = store.search(query_vector, cfg.RETRIEVAL_TOP_K, score_threshold=cfg.HARD_CUTOFF)

    context_str = ""
    citations: list[str] = []
    citation_meta: list[dict] = []
    top_score = 0.0
    for res in results:
        top_score = max(top_score, res.score)
        node_id = res.payload["kg_node_id"]
        citations.append(node_id)
        citation_meta.append({
            "id": node_id, "title": res.payload["title"], "chapter": res.payload.get("chapter", ""),
            "score": round(res.score, 3),
        })
        graph_rows = store.graph_context(node_id)
        graph_info = "\n".join(f"  - {row['type']}: {row['name']}" for row in graph_rows)
        context_str += f"\n[{node_id}] {res.payload['text'][:cfg.CONTEXT_CHAR_LIMIT]}\n{graph_info}\n"

    logger.info(
        "Retrieval for %r: %d results above HARD_CUTOFF=%.2f, top_score=%.3f",
        query, len(results), cfg.HARD_CUTOFF, top_score,
    )
    return context_str, citations, citation_meta, top_score


def _stream_answer(query: str):
    """Generator yielding NDJSON lines: token chunks, then a final summary line."""
    run_ingestion()

    if _ingestion_error:
        yield _ndjson({
            "type": "done", "status": "no_answer",
            "note": f"The Act text could not be loaded ({_ingestion_error}). "
                    f"This will retry automatically on the next question.",
            "citations": [], "citation_meta": [],
        })
        return

    cached = answer_cache.get(query)
    if cached:
        yield _ndjson({"type": "token", "text": cached["answer"]})
        yield _ndjson({
            "type": "done", "status": "answered",
            "citations": cached["citations"], "citation_meta": cached["citation_meta"], "cached": True,
        })
        return

    context_str, citations, citation_meta, top_score = _retrieve(query)

    if not citations:
        # Two genuinely different situations, worth telling the user apart:
        if store.indexed_count() == 0:
            note = "Still finishing ingestion of the Act text - please try again in a moment."
        else:
            note = (
                "I couldn't find anything about this in the DPDP Act, 2023. This assistant "
                "only answers questions about this specific Act, not other laws or general "
                "data-privacy topics."
            )
        yield _ndjson({
            "type": "done", "status": "no_answer", "note": note,
            "citations": [], "citation_meta": [],
        })
        return

    if query_is_high_risk(query):
        yield _ndjson({
            "type": "done", "status": "pending_review",
            "note": "High-risk question (penalty/obligation/cross-border). Held for human "
                    "review instead of an auto-generated answer.",
            "citations": citations, "citation_meta": citation_meta,
        })
        return

    low_confidence = top_score < cfg.SIMILARITY_THRESHOLD
    user_prompt = f"Context from the DPDP Act, 2023:\n{context_str}\n\nQuestion: {query}"

    full_answer = ""
    try:
        for chunk in ollama.chat(
            model=cfg.OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            options={"num_predict": cfg.OLLAMA_NUM_PREDICT, "num_ctx": cfg.OLLAMA_NUM_CTX},
            keep_alive=cfg.OLLAMA_KEEP_ALIVE,
            stream=True,
        ):
            piece = chunk.get("message", {}).get("content", "")
            if piece:
                full_answer += piece
                yield _ndjson({"type": "token", "text": piece})
    except Exception as exc:  # local model failed mid-stream — surface it, don't crash the app
        logger.exception("Ollama generation failed")
        yield _ndjson({"type": "error", "detail": str(exc)})
        return

    if low_confidence:
        disclaimer = "\n\n(Low-confidence match - please verify against the cited section text.)"
        full_answer += disclaimer
        yield _ndjson({"type": "token", "text": disclaimer})

    answer_cache.set(query, {
        "answer": full_answer, "citations": citations, "citation_meta": citation_meta,
    })
    yield _ndjson({
        "type": "done", "status": "answered",
        "citations": citations, "citation_meta": citation_meta, "cached": False,
    })


@app.post("/ask")
def ask_dpdp(request: QueryRequest, http_request: Request):
    client_key = http_request.client.host if http_request.client else "unknown"
    if not rate_limiter.allow(client_key):
        raise HTTPException(status_code=429, detail="Too many requests - please slow down.")
    query = request.clean_query()
    return StreamingResponse(_stream_answer(query), media_type="application/x-ndjson")


# ---------------------------------------------------------------------------
# Frontend (single-page app, served from the same process/port)
# ---------------------------------------------------------------------------
app.mount("/", StaticFiles(directory="static", html=True), name="static")


## 9. Write the production frontend (hand-built HTML/CSS/JS, no framework, no build step)

A real chat application shell: sidebar with live backend status, indexed/pending counters, and the
human-review queue (approve/reject buttons update instantly); a main chat panel with streamed
responses and a typing indicator; citation chips that open the actual section text in a modal; and a
mobile-responsive layout. No paid fonts, icons, or APIs - the only external resource is the
open-source Inter/JetBrains Mono font pair from Google Fonts, loaded by the end user's own browser.

In [ ]:
import os
os.makedirs("static", exist_ok=True)
print("static/ ready")


In [ ]:
%%writefile static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>DPDP Act Assistant</title>
<link rel="icon" href="data:image/svg+xml,<svg xmlns=%22http://www.w3.org/2000/svg%22 viewBox=%220 0 24 24%22><text y=%2220%22 font-size=%2220%22>⚖️</text></svg>" />
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<link rel="stylesheet" href="/style.css" />
</head>
<body>

<div class="app">

  <!-- ============ SIDEBAR ============ -->
  <aside class="sidebar" id="sidebar">
    <div class="brand">
      <div class="brand-mark">⚖️</div>
      <div class="brand-text">
        <div class="brand-title">DPDP Act Assistant</div>
        <div class="brand-sub">GraphRAG · Human-in-the-loop</div>
      </div>
    </div>

    <div class="status-row" id="statusRow">
      <span class="status-dot" id="statusDot"></span>
      <span id="statusText">Connecting…</span>
    </div>

    <div class="stat-grid" id="statGrid">
      <div class="stat-card">
        <div class="stat-value" id="statIndexed">–</div>
        <div class="stat-label">Indexed</div>
      </div>
      <div class="stat-card">
        <div class="stat-value" id="statPending">–</div>
        <div class="stat-label">In review</div>
      </div>
    </div>

    <div class="sidebar-section">
      <div class="sidebar-heading">
        <span>Human review queue</span>
        <span class="badge" id="reviewBadge">0</span>
      </div>
      <p class="sidebar-hint">
        Sections touching an obligation or penalty, or with low parse confidence, are held here.
        Nothing below is searchable until approved.
      </p>
      <div class="review-list" id="reviewList">
        <div class="empty-state">Loading queue…</div>
      </div>
    </div>

    <div class="sidebar-footer">
      Source: DPDP Act, 2023 · Ministry of Electronics &amp; IT (MeitY)
      <br/>Answers generated by a local open-source model — always verify against the cited section.
    </div>
  </aside>

  <button class="sidebar-toggle" id="sidebarToggle" aria-label="Toggle sidebar">☰</button>

  <!-- ============ MAIN CHAT ============ -->
  <main class="main">
    <header class="chat-header">
      <div>
        <h1>Ask about the DPDP Act, 2023</h1>
        <p>Grounded answers with citations, backed by a knowledge graph + vector index.</p>
      </div>
    </header>

    <div class="chat-scroll" id="chatScroll">
      <div class="welcome" id="welcomeCard">
        <div class="welcome-icon">⚖️</div>
        <h2>Ask a question to get started</h2>
        <p>Every answer is grounded in the official Act text and shows exactly which sections it drew from.</p>
        <div class="suggestion-row">
          <button class="suggestion-chip" data-q="What are the grounds for processing personal data?">Grounds for processing personal data</button>
          <button class="suggestion-chip" data-q="What rights does a Data Principal have?">Rights of a Data Principal</button>
          <button class="suggestion-chip" data-q="What penalty applies for a data breach?">Penalty for a data breach</button>
        </div>
      </div>
      <div id="messages"></div>
    </div>

    <form class="composer" id="composerForm">
      <input
        id="composerInput"
        type="text"
        maxlength="500"
        autocomplete="off"
        placeholder="Ask about obligations, rights, penalties, or definitions…"
      />
      <button type="submit" id="sendBtn" aria-label="Send">
        <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M3 12L21 3L14 21L11 13L3 12Z" stroke="currentColor" stroke-width="2" stroke-linejoin="round"/></svg>
      </button>
    </form>
  </main>
</div>

<!-- ============ CITATION MODAL ============ -->
<div class="modal-backdrop" id="modalBackdrop">
  <div class="modal">
    <div class="modal-header">
      <span id="modalTitle">Section</span>
      <button class="modal-close" id="modalClose">✕</button>
    </div>
    <div class="modal-body" id="modalBody"></div>
  </div>
</div>

<script src="/app.js"></script>
</body>
</html>


In [ ]:
%%writefile static/style.css
:root {
  --bg: #0b0d12;
  --bg-elevated: #12151c;
  --bg-card: #161a23;
  --border: #232834;
  --border-soft: #1b1f29;
  --text: #e7e9ee;
  --text-dim: #9aa1b2;
  --text-faint: #5f6577;
  --accent: #6d8cff;
  --accent-soft: rgba(109, 140, 255, 0.14);
  --accent-strong: #829bff;
  --success: #3ecf8e;
  --warn: #f5b93d;
  --danger: #f26d6d;
  --radius-lg: 16px;
  --radius-md: 10px;
  --radius-sm: 7px;
  --shadow-lg: 0 20px 48px rgba(0, 0, 0, 0.45);
  --font: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
  --mono: 'JetBrains Mono', ui-monospace, monospace;
}

* { box-sizing: border-box; }

html, body {
  height: 100%;
  margin: 0;
  background: var(--bg);
  color: var(--text);
  font-family: var(--font);
  -webkit-font-smoothing: antialiased;
}

.app {
  display: grid;
  grid-template-columns: 300px 1fr;
  height: 100vh;
  overflow: hidden;
}

/* ============ SIDEBAR ============ */
.sidebar {
  background: var(--bg-elevated);
  border-right: 1px solid var(--border-soft);
  display: flex;
  flex-direction: column;
  padding: 20px 18px;
  overflow-y: auto;
  gap: 18px;
}

.brand { display: flex; align-items: center; gap: 12px; }
.brand-mark {
  width: 38px; height: 38px;
  display: flex; align-items: center; justify-content: center;
  background: linear-gradient(135deg, var(--accent), #4b63cc);
  border-radius: var(--radius-md);
  font-size: 18px;
  flex-shrink: 0;
}
.brand-title { font-weight: 700; font-size: 15px; letter-spacing: -0.01em; }
.brand-sub { font-size: 11.5px; color: var(--text-faint); margin-top: 2px; }

.status-row {
  display: flex; align-items: center; gap: 8px;
  font-size: 12.5px; color: var(--text-dim);
  background: var(--bg-card);
  border: 1px solid var(--border-soft);
  padding: 8px 12px;
  border-radius: var(--radius-sm);
}
.status-dot {
  width: 8px; height: 8px; border-radius: 50%;
  background: var(--text-faint);
  flex-shrink: 0;
  transition: background 0.2s;
}
.status-dot.online { background: var(--success); box-shadow: 0 0 8px rgba(62, 207, 142, 0.6); }
.status-dot.offline { background: var(--danger); box-shadow: 0 0 8px rgba(242, 109, 109, 0.5); }

.stat-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }
.stat-card {
  background: var(--bg-card);
  border: 1px solid var(--border-soft);
  border-radius: var(--radius-md);
  padding: 12px;
}
.stat-value { font-size: 20px; font-weight: 700; font-family: var(--mono); }
.stat-label { font-size: 11px; color: var(--text-faint); margin-top: 2px; text-transform: uppercase; letter-spacing: 0.04em; }

.sidebar-section { display: flex; flex-direction: column; gap: 10px; flex: 1; min-height: 0; }
.sidebar-heading {
  display: flex; align-items: center; justify-content: space-between;
  font-size: 12.5px; font-weight: 600; color: var(--text-dim);
  text-transform: uppercase; letter-spacing: 0.03em;
}
.badge {
  background: var(--accent-soft); color: var(--accent-strong);
  font-size: 11px; font-weight: 700; padding: 1px 8px; border-radius: 999px;
  font-family: var(--mono);
}
.sidebar-hint { font-size: 12px; color: var(--text-faint); line-height: 1.5; margin: 0; }

.review-list { display: flex; flex-direction: column; gap: 8px; overflow-y: auto; padding-right: 2px; }
.empty-state { font-size: 12.5px; color: var(--text-faint); padding: 14px 4px; text-align: center; }

.review-item {
  background: var(--bg-card);
  border: 1px solid var(--border-soft);
  border-radius: var(--radius-md);
  padding: 10px 12px;
}
.review-item-title { font-size: 13px; font-weight: 600; margin-bottom: 4px; }
.review-item-meta { font-size: 11px; color: var(--text-faint); display: flex; gap: 10px; margin-bottom: 8px; }
.review-item-meta .flag { color: var(--warn); }
.confidence-bar { height: 4px; border-radius: 2px; background: var(--border); overflow: hidden; margin-bottom: 8px; }
.confidence-fill { height: 100%; background: var(--accent); }
.review-actions { display: flex; gap: 6px; }
.btn-mini {
  flex: 1; font-size: 11.5px; font-weight: 600; padding: 6px 0;
  border-radius: 7px; border: 1px solid var(--border); cursor: pointer;
  background: transparent; color: var(--text-dim); transition: all 0.15s;
}
.btn-mini.approve:hover { background: rgba(62, 207, 142, 0.12); border-color: var(--success); color: var(--success); }
.btn-mini.reject:hover { background: rgba(242, 109, 109, 0.12); border-color: var(--danger); color: var(--danger); }
.btn-mini:disabled { opacity: 0.5; cursor: default; }

.sidebar-footer { font-size: 10.5px; color: var(--text-faint); line-height: 1.5; padding-top: 8px; border-top: 1px solid var(--border-soft); }

.sidebar-toggle {
  display: none;
  position: fixed; top: 14px; left: 14px; z-index: 20;
  width: 36px; height: 36px; border-radius: 9px;
  background: var(--bg-card); border: 1px solid var(--border);
  color: var(--text); font-size: 16px; cursor: pointer;
}

/* ============ MAIN ============ */
.main { display: flex; flex-direction: column; height: 100vh; min-width: 0; }

.chat-header {
  padding: 22px 32px 16px;
  border-bottom: 1px solid var(--border-soft);
}
.chat-header h1 { font-size: 18px; font-weight: 700; margin: 0 0 4px; letter-spacing: -0.01em; }
.chat-header p { font-size: 13px; color: var(--text-faint); margin: 0; }

.chat-scroll { flex: 1; overflow-y: auto; padding: 24px 32px; }

.welcome { max-width: 640px; margin: 40px auto; text-align: center; }
.welcome-icon { font-size: 34px; margin-bottom: 12px; }
.welcome h2 { font-size: 19px; margin: 0 0 8px; }
.welcome p { color: var(--text-faint); font-size: 13.5px; line-height: 1.6; margin: 0 0 22px; }
.suggestion-row { display: flex; flex-wrap: wrap; gap: 8px; justify-content: center; }
.suggestion-chip {
  background: var(--bg-card); border: 1px solid var(--border);
  color: var(--text-dim); font-size: 12.5px; padding: 9px 14px;
  border-radius: 999px; cursor: pointer; font-family: var(--font);
  transition: all 0.15s;
}
.suggestion-chip:hover { border-color: var(--accent); color: var(--accent-strong); background: var(--accent-soft); }

#messages { display: flex; flex-direction: column; gap: 18px; max-width: 760px; margin: 0 auto; }

.msg { display: flex; gap: 12px; animation: fadeIn 0.25s ease; }
@keyframes fadeIn { from { opacity: 0; transform: translateY(4px); } to { opacity: 1; transform: translateY(0); } }

.msg-avatar {
  width: 30px; height: 30px; border-radius: 8px;
  display: flex; align-items: center; justify-content: center;
  font-size: 14px; flex-shrink: 0; margin-top: 2px;
}
.msg.user .msg-avatar { background: var(--border); }
.msg.assistant .msg-avatar { background: linear-gradient(135deg, var(--accent), #4b63cc); }

.msg-body { flex: 1; min-width: 0; }
.msg-role { font-size: 12px; font-weight: 600; color: var(--text-faint); margin-bottom: 5px; }
.msg-content {
  font-size: 14.5px; line-height: 1.65; color: var(--text);
  white-space: pre-wrap; word-wrap: break-word;
}
.msg.user .msg-content { color: var(--text-dim); }

.msg-content.status-pending { color: var(--warn); }
.msg-content.status-error { color: var(--danger); }
.msg-content.status-empty { color: var(--text-faint); }

.citations { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 10px; }
.citation-chip {
  font-family: var(--mono); font-size: 11.5px;
  background: var(--accent-soft); color: var(--accent-strong);
  border: 1px solid rgba(109, 140, 255, 0.25);
  padding: 3px 10px; border-radius: 999px; cursor: pointer;
  transition: all 0.15s;
}
.citation-chip:hover { background: rgba(109, 140, 255, 0.24); }
.cache-tag { font-size: 10.5px; color: var(--text-faint); margin-top: 8px; font-family: var(--mono); }

.typing-dots { display: inline-flex; gap: 3px; padding: 2px 0; }
.typing-dots span {
  width: 6px; height: 6px; border-radius: 50%; background: var(--text-faint);
  animation: bounce 1.3s infinite ease-in-out;
}
.typing-dots span:nth-child(2) { animation-delay: 0.15s; }
.typing-dots span:nth-child(3) { animation-delay: 0.3s; }
@keyframes bounce { 0%, 60%, 100% { transform: translateY(0); opacity: 0.4; } 30% { transform: translateY(-4px); opacity: 1; } }

.composer {
  display: flex; gap: 10px; align-items: center;
  padding: 16px 32px 22px;
  border-top: 1px solid var(--border-soft);
}
.composer input {
  flex: 1; background: var(--bg-card); border: 1px solid var(--border);
  color: var(--text); font-family: var(--font); font-size: 14px;
  padding: 13px 16px; border-radius: 12px; outline: none;
  transition: border-color 0.15s;
}
.composer input:focus { border-color: var(--accent); }
.composer input::placeholder { color: var(--text-faint); }
.composer button {
  width: 44px; height: 44px; border-radius: 12px; border: none;
  background: var(--accent); color: #fff; cursor: pointer;
  display: flex; align-items: center; justify-content: center;
  flex-shrink: 0; transition: background 0.15s, transform 0.1s;
}
.composer button:hover { background: var(--accent-strong); }
.composer button:active { transform: scale(0.94); }
.composer button:disabled { background: var(--border); cursor: default; }

/* ============ MODAL ============ */
.modal-backdrop {
  position: fixed; inset: 0; background: rgba(5, 6, 10, 0.65);
  display: none; align-items: center; justify-content: center; z-index: 50;
  backdrop-filter: blur(2px);
}
.modal-backdrop.open { display: flex; }
.modal {
  width: min(560px, 92vw); max-height: 78vh; display: flex; flex-direction: column;
  background: var(--bg-elevated); border: 1px solid var(--border);
  border-radius: var(--radius-lg); box-shadow: var(--shadow-lg);
}
.modal-header {
  display: flex; align-items: center; justify-content: space-between;
  padding: 16px 20px; border-bottom: 1px solid var(--border-soft);
  font-weight: 600; font-size: 14.5px;
}
.modal-close { background: none; border: none; color: var(--text-faint); font-size: 15px; cursor: pointer; }
.modal-body { padding: 18px 20px; overflow-y: auto; font-size: 13.5px; line-height: 1.7; color: var(--text-dim); white-space: pre-wrap; }

/* ============ RESPONSIVE ============ */
@media (max-width: 860px) {
  .app { grid-template-columns: 1fr; }
  .sidebar {
    position: fixed; inset: 0 30% 0 0; z-index: 15;
    transform: translateX(-100%); transition: transform 0.2s ease;
    box-shadow: var(--shadow-lg);
  }
  .sidebar.open { transform: translateX(0); }
  .sidebar-toggle { display: flex; align-items: center; justify-content: center; }
  .chat-header, .chat-scroll, .composer { padding-left: 18px; padding-right: 18px; }
  .chat-header { padding-top: 56px; }
}


In [ ]:
%%writefile static/app.js
const API = "";

const el = {
  statusDot: document.getElementById("statusDot"),
  statusText: document.getElementById("statusText"),
  statIndexed: document.getElementById("statIndexed"),
  statPending: document.getElementById("statPending"),
  reviewBadge: document.getElementById("reviewBadge"),
  reviewList: document.getElementById("reviewList"),
  chatScroll: document.getElementById("chatScroll"),
  messages: document.getElementById("messages"),
  welcomeCard: document.getElementById("welcomeCard"),
  composerForm: document.getElementById("composerForm"),
  composerInput: document.getElementById("composerInput"),
  sendBtn: document.getElementById("sendBtn"),
  sidebar: document.getElementById("sidebar"),
  sidebarToggle: document.getElementById("sidebarToggle"),
  modalBackdrop: document.getElementById("modalBackdrop"),
  modalTitle: document.getElementById("modalTitle"),
  modalBody: document.getElementById("modalBody"),
  modalClose: document.getElementById("modalClose"),
};

let isStreaming = false;

// ---------------------------------------------------------------------------
// Sidebar toggle (mobile)
// ---------------------------------------------------------------------------
el.sidebarToggle.addEventListener("click", () => el.sidebar.classList.toggle("open"));

// ---------------------------------------------------------------------------
// Health + stats + review queue polling
// ---------------------------------------------------------------------------
async function refreshHealth() {
  try {
    const res = await fetch(`${API}/health`);
    const data = await res.json();
    if (data.ingestion_error) {
      el.statusDot.className = "status-dot offline";
      el.statusText.textContent = `Ingestion failed: ${data.ingestion_error}`;
    } else if (data.ingested) {
      el.statusDot.className = "status-dot online";
      el.statusText.textContent = "Backend online";
    } else {
      el.statusDot.className = "status-dot";
      el.statusText.textContent = "Ingesting Act text…";
    }
  } catch {
    el.statusDot.className = "status-dot offline";
    el.statusText.textContent = "Backend unreachable";
  }
}

async function refreshStats() {
  try {
    const res = await fetch(`${API}/stats`);
    const data = await res.json();
    el.statIndexed.textContent = data.indexed_in_qdrant ?? "–";
    el.statPending.textContent = data.pending_review ?? "–";
    el.reviewBadge.textContent = data.pending_review ?? "0";
  } catch { /* backend still starting */ }
}

async function refreshReviewQueue() {
  try {
    const res = await fetch(`${API}/pending-review`);
    const items = await res.json();
    renderReviewQueue(items);
  } catch { /* backend still starting */ }
}

function renderReviewQueue(items) {
  if (!items.length) {
    el.reviewList.innerHTML = `<div class="empty-state">No sections waiting for review right now.</div>`;
    return;
  }
  el.reviewList.innerHTML = "";
  for (const item of items) {
    const card = document.createElement("div");
    card.className = "review-item";
    const pct = Math.round(item.confidence * 100);
    card.innerHTML = `
      <div class="review-item-title">${item.section_id} · ${escapeHtml(item.title)}</div>
      <div class="review-item-meta">
        <span>Confidence ${pct}%</span>
        ${item.sensitive ? `<span class="flag">⚠ sensitive</span>` : ""}
      </div>
      <div class="confidence-bar"><div class="confidence-fill" style="width:${pct}%"></div></div>
      <div class="review-actions">
        <button class="btn-mini approve">Approve</button>
        <button class="btn-mini reject">Reject</button>
      </div>
    `;
    const [approveBtn, rejectBtn] = card.querySelectorAll("button");
    approveBtn.addEventListener("click", () => decide(item.section_id, "approve", card));
    rejectBtn.addEventListener("click", () => decide(item.section_id, "reject", card));
    el.reviewList.appendChild(card);
  }
}

async function decide(sectionId, decision, cardEl) {
  cardEl.querySelectorAll("button").forEach((b) => (b.disabled = true));
  try {
    await fetch(`${API}/approve-review-item`, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ section_id: sectionId, decision, reviewer: "demo_reviewer" }),
    });
    await Promise.all([refreshReviewQueue(), refreshStats()]);
  } catch {
    cardEl.querySelectorAll("button").forEach((b) => (b.disabled = false));
  }
}

refreshHealth();
refreshStats();
refreshReviewQueue();
setInterval(refreshHealth, 8000);
setInterval(() => { refreshStats(); refreshReviewQueue(); }, 6000);

// ---------------------------------------------------------------------------
// Chat
// ---------------------------------------------------------------------------
function escapeHtml(str) {
  const d = document.createElement("div");
  d.textContent = str;
  return d.innerHTML;
}

function scrollToBottom() {
  el.chatScroll.scrollTop = el.chatScroll.scrollHeight;
}

function addMessage(role) {
  el.welcomeCard.style.display = "none";
  const wrap = document.createElement("div");
  wrap.className = `msg ${role}`;
  wrap.innerHTML = `
    <div class="msg-avatar">${role === "user" ? "🧑" : "⚖️"}</div>
    <div class="msg-body">
      <div class="msg-role">${role === "user" ? "You" : "DPDP Assistant"}</div>
      <div class="msg-content"></div>
    </div>
  `;
  el.messages.appendChild(wrap);
  scrollToBottom();
  return wrap.querySelector(".msg-content");
}

function showTyping(contentEl) {
  contentEl.innerHTML = `<span class="typing-dots"><span></span><span></span><span></span></span>`;
}

async function sendQuery(query) {
  if (isStreaming) return;
  isStreaming = true;
  el.sendBtn.disabled = true;

  addMessage("user").textContent = query;
  const assistantContent = addMessage("assistant");
  showTyping(assistantContent);

  let firstToken = true;
  let citationMeta = [];

  try {
    const res = await fetch(`${API}/ask`, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ query }),
    });

    if (res.status === 429) {
      assistantContent.className = "msg-content status-error";
      assistantContent.textContent = "You're sending questions a bit fast — please wait a moment and try again.";
      return;
    }
    if (!res.ok || !res.body) {
      assistantContent.className = "msg-content status-error";
      assistantContent.textContent = `Request failed (${res.status}).`;
      return;
    }

    const reader = res.body.getReader();
    const decoder = new TextDecoder();
    let buffer = "";

    while (true) {
      const { value, done } = await reader.read();
      if (done) break;
      buffer += decoder.decode(value, { stream: true });
      const lines = buffer.split("\n");
      buffer = lines.pop(); // keep the last (possibly incomplete) line

      for (const line of lines) {
        if (!line.trim()) continue;
        const evt = JSON.parse(line);

        if (evt.type === "token") {
          if (firstToken) { assistantContent.textContent = ""; firstToken = false; }
          assistantContent.textContent += evt.text;
          scrollToBottom();
        } else if (evt.type === "error") {
          assistantContent.className = "msg-content status-error";
          assistantContent.textContent = `The local model hit an error: ${evt.detail}`;
        } else if (evt.type === "done") {
          citationMeta = evt.citation_meta || [];
          if (evt.status === "no_answer") {
            assistantContent.className = "msg-content status-empty";
            assistantContent.textContent = evt.note;
          } else if (evt.status === "pending_review") {
            assistantContent.className = "msg-content status-pending";
            assistantContent.textContent = evt.note;
          } else if (firstToken) {
            // shouldn't normally happen, but guards against an empty stream
            assistantContent.textContent = "(no content generated)";
          }
          renderCitations(assistantContent, citationMeta, evt.cached);
        }
      }
    }
  } catch (err) {
    assistantContent.className = "msg-content status-error";
    assistantContent.textContent = "Connection lost while generating the answer. Please try again.";
  } finally {
    isStreaming = false;
    el.sendBtn.disabled = false;
    scrollToBottom();
  }
}

function renderCitations(afterEl, citationMeta, cached) {
  if (!citationMeta.length) return;
  const row = document.createElement("div");
  row.className = "citations";
  for (const c of citationMeta) {
    const chip = document.createElement("button");
    chip.className = "citation-chip";
    chip.type = "button";
    chip.textContent = `${c.id} · ${c.title}`;
    chip.addEventListener("click", () => openCitation(c.id));
    row.appendChild(chip);
  }
  afterEl.insertAdjacentElement("afterend", row);
  if (cached) {
    const tag = document.createElement("div");
    tag.className = "cache-tag";
    tag.textContent = "served from cache";
    row.insertAdjacentElement("afterend", tag);
  }
}

async function openCitation(sectionId) {
  el.modalTitle.textContent = "Loading…";
  el.modalBody.textContent = "";
  el.modalBackdrop.classList.add("open");
  try {
    const res = await fetch(`${API}/section/${sectionId}`);
    const data = await res.json();
    el.modalTitle.textContent = `${data.id} · ${data.title}`;
    el.modalBody.textContent = data.text;
  } catch {
    el.modalTitle.textContent = "Error";
    el.modalBody.textContent = "Could not load this section.";
  }
}

el.modalClose.addEventListener("click", () => el.modalBackdrop.classList.remove("open"));
el.modalBackdrop.addEventListener("click", (e) => {
  if (e.target === el.modalBackdrop) el.modalBackdrop.classList.remove("open");
});

el.composerForm.addEventListener("submit", (e) => {
  e.preventDefault();
  const q = el.composerInput.value.trim();
  if (!q) return;
  el.composerInput.value = "";
  sendQuery(q);
});

document.querySelectorAll(".suggestion-chip").forEach((chip) => {
  chip.addEventListener("click", () => sendQuery(chip.dataset.q));
});


## 10. Launch the backend (serves the API *and* the frontend on one port)

Only one process to manage now - no separate Streamlit server, no cross-process log juggling. Logs
go to `uvicorn.log`. Startup should take well under a minute (no LLM calls happen until you actually
ask a question).

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn"])
time.sleep(2)

uvicorn_log = open("uvicorn.log", "w")

print("Starting the app (API + frontend) on port 8000 (logs -> uvicorn.log)...")
subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=uvicorn_log, stderr=subprocess.STDOUT,
)

print("Launched. Startup is fast (fetch + parse + rule-based tagging, no LLM calls) - should be")
print("ready in under a minute. Next cell waits for it and shows live progress.")


## 11. Wait for the backend, then test it (inside Colab, no tunnel needed yet)

Polls `/health`, and prints the tail of `uvicorn.log` every 15 seconds so you can see actual
progress instead of a silent wait. Also exercises the new streaming `/ask` endpoint directly, so you
can see the NDJSON chunks the frontend is parsing.

In [ ]:
import requests as _requests
import os
import time as _time
import json as _json


def tail_log(path, n=60):
    if not os.path.exists(path):
        return "(no log file yet)"
    with open(path) as f:
        lines = f.readlines()
    return "".join(lines[-n:]) if lines else "(log file is empty so far)"


def wait_for_backend(max_wait_seconds=180, poll_interval=5):
    waited = 0
    while waited < max_wait_seconds:
        try:
            r = _requests.get("http://localhost:8000/health", timeout=5)
            if r.status_code == 200:
                print(f"Backend is up after about {waited}s.")
                return True
        except _requests.exceptions.RequestException:
            pass
        if waited % 15 == 0:
            print(f"  ({waited}s elapsed) latest log lines:")
            print("    " + tail_log("uvicorn.log", 3).replace("\n", "\n    "))
        _time.sleep(poll_interval)
        waited += poll_interval
    return False


def ask_streaming(query, timeout=300):
    """Calls /ask and prints each NDJSON chunk as it arrives - exactly what the browser does."""
    full_answer = ""
    with _requests.post("http://localhost:8000/ask", json={"query": query}, stream=True, timeout=timeout) as r:
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            evt = _json.loads(line)
            if evt["type"] == "token":
                full_answer += evt["text"]
            elif evt["type"] == "done":
                print(f"  -> status={evt['status']}  citations={evt.get('citations')}")
    return full_answer


print("Waiting for the backend (should be well under a minute now)...")
if not wait_for_backend():
    print()
    print("Backend did not come up in time. Full uvicorn.log:")
    print("-" * 70)
    print(tail_log("uvicorn.log"))
    print("-" * 70)
    print("Paste this log back to fix the actual cause - re-running the launch cell then this cell after a fix.")
else:
    print()
    print("Health check:")
    print(_requests.get("http://localhost:8000/health", timeout=10).json())

    print()
    print("Stats:")
    print(_requests.get("http://localhost:8000/stats", timeout=10).json())

    print()
    print("Normal question (streamed):")
    answer = ask_streaming("What are the grounds for processing personal data?")
    print(f"  answer: {answer}")

    print()
    print("High-risk question (should come back pending_review, not a final answer):")
    answer = ask_streaming("What penalty applies if a company fails to report a data breach?")
    print(f"  answer: {answer!r} (empty is expected - it was held for review)")


## 12. Open a public URL (free, no signup - `cloudflared`)

Only one tunnel needed now, since the frontend and API share a port. `cloudflared`'s free quick
tunnels have no interstitial warning page or password step at all - the URL just works.

In [ ]:
import re as _re

print("Opening a Cloudflare quick tunnel to the app on port 8000...")
tunnel_log = open("cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    with open("cloudflared.log") as f:
        log_content = f.read()
    match = _re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"Your app is live at: {public_url}")
    print("No password, no signup - open it directly. This process keeps running in the")
    print("background to keep the tunnel open; you do not need to leave this cell running.")
else:
    print("Tunnel URL not found yet - here is cloudflared.log so far:")
    with open("cloudflared.log") as f:
        print(f.read())
    print("Re-run this cell if the URL did not appear - it usually takes a few seconds.")


## Project summary

**What's real in this demo:** the DPDP Act text, fetched live from the official government PDF every
time this notebook runs (not hardcoded); a real Neo4j graph and real Qdrant vector index; a real
local LLM answering questions, streamed token-by-token; a working human-review gate that actually
blocks unapproved content from being searchable and updates live in the UI; a hand-built production
frontend with no framework, build step, or paid dependency; and a public, shareable URL.

**What's simplified, and clearly marked as such throughout:** rule-based rather than LLM-based entity
tagging (`dpdp_ingest.py`), a single Act rather than all 70+ jurisdictions in the original project
scope, and ephemeral rather than persistent infrastructure. None of these are hidden - each is
called out at the point in the notebook where it matters.

## Where this would grow for a production system

1. Replace rule-based tagging with a real LLM extraction agent run as an **offline batch job**
   (not blocking server startup) - the review-queue mechanics here plug in directly, only the
   source of the confidence score changes.
2. Persist Neo4j and Qdrant outside the Colab container - both live in `/content` here and are
   wiped when the session ends.
3. Also ingest the **Digital Personal Data Protection Rules, 2025** (notified 13 November 2025) -
   this only covers the original 2023 Act text.
4. Add amendment monitoring: stage detected law changes for human confirmation before they update
   the graph, same `review_queue` pattern extended.
5. Repeat this same pattern per country to build out the full multi-jurisdiction graph described in
   the original project scope (70+ jurisdictions, DPDP/GDPR/CCPA/LGPD/PIPL and others).
6. Swap `qwen2.5:3b` for a larger Ollama model (e.g. `qwen2.5:7b`, `llama3.1:8b`) if answer precision
   needs to improve - the "Section 41 vs Section 42" mislabeling example above is a small-model
   limitation, not an architectural one.
7. Move the in-memory `ReviewQueue`, `AnswerCache`, and `RateLimiter` (`dpdp_stores.py`) to Redis or
   a small database once this runs across more than one process/worker.
